# 03 — National ASOS Training Data Pull

Downloads ASOS data for national stations (2018–2023), assembles the training parquet,
and pulls DC metro test data (2024–present).

**Years 2018–2021 already downloaded.** This notebook will skip those and only pull 2022–2023.
Then it combines all 6 years and creates the DC metro test set.

In [ ]:
import sys
sys.path.insert(0, '/Users/dengzhenhua/Library/Python/3.9/lib/python/site-packages')

BASE_DIR = '/Users/dengzhenhua/Desktop/Desktop - MacBook Pro/work/vibe coding/fogchaser'
RANDOM_SEED = 42

In [ ]:
# Step 1: Define national station list (excluding DC metro)
import requests
import pandas as pd
import io
import os
import time

DC_STATIONS = ['KDCA', 'KIAD', 'KBWI', 'KGAI', 'KFDK', 'KHEF', 'KNYG', 'KCGS']

TARGET_NETWORKS = [
    'MD_ASOS', 'VA_ASOS', 'DC_ASOS', 'WV_ASOS', 'PA_ASOS',
    'NC_ASOS', 'TN_ASOS', 'KY_ASOS',
    'OH_ASOS', 'IN_ASOS', 'IL_ASOS', 'MO_ASOS',
    'OR_ASOS', 'WA_ASOS', 'CA_ASOS',
]

def get_stations_for_network(network):
    url = f"https://mesonet.agron.iastate.edu/geojson/network/{network}.geojson"
    response = requests.get(url, timeout=30)
    data = response.json()
    stations = [f['properties']['sid'] for f in data['features']]
    return stations

all_stations = []
for network in TARGET_NETWORKS:
    stations = get_stations_for_network(network)
    all_stations.extend(stations)
    print(f"{network}: {len(stations)} stations")

all_stations = list(set(all_stations))
before_count = len(all_stations)
all_stations = [s for s in all_stations if s not in DC_STATIONS]
after_count = len(all_stations)
print(f"\nTotal unique stations: {before_count}")
print(f"After removing DC metro holdout: {after_count}")

In [ ]:
# Step 2: Pull ASOS data (2018–2023) — skips years already downloaded

def fetch_iem_asos(station, year_start, year_end):
    url = (
        f"https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"
        f"?station={station}"
        f"&data=tmpf&data=dwpf&data=sknt&data=drct&data=vsby&data=wxcodes&data=metar"
        f"&year1={year_start}&month1=1&day1=1"
        f"&year2={year_end}&month2=12&day2=31"
        f"&tz=Etc/UTC&format=onlycomma&latlon=yes&elev=yes&direct=no&report_type=3"
    )
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    df = pd.read_csv(io.StringIO(response.text),
                     parse_dates=['valid'],
                     na_values=['M', 'T', ''])
    df['station'] = station
    return df

def pull_year_for_stations(stations, year, output_dir):
    out_path = f'{output_dir}/asos_national_{year}.parquet'
    if os.path.exists(out_path):
        print(f"{year} already exists, skipping.")
        return

    year_frames = []
    for i, station in enumerate(stations):
        try:
            df = fetch_iem_asos(station, year, year)
            if len(df) > 100:
                year_frames.append(df)
        except Exception as e:
            print(f"  Error {station}: {e}")
        if i % 50 == 0:
            print(f"  {i}/{len(stations)} stations done...")
        time.sleep(0.1)

    combined = pd.concat(year_frames, ignore_index=True)
    combined.to_parquet(out_path)
    print(f"Saved {len(combined):,} rows for {year}.")

for year in range(2018, 2024):
    print(f"\n--- Pulling {year} ---")
    pull_year_for_stations(all_stations, year, f'{BASE_DIR}/data/raw/asos')

In [ ]:
# Step 3: Combine all years and create fog labels
import numpy as np

frames = []
for year in range(2018, 2024):
    path = f'{BASE_DIR}/data/raw/asos/asos_national_{year}.parquet'
    assert os.path.exists(path), f"Missing year file: {path}"
    df = pd.read_parquet(path)
    frames.append(df)
    print(f"{year}: {len(df):,} rows loaded")

national_df = pd.concat(frames, ignore_index=True)
national_df['valid'] = pd.to_datetime(national_df['valid'], utc=True)
national_df['vsby_km'] = national_df['vsby'] * 1.60934
national_df['is_fog'] = (national_df['vsby_km'] < 1.0).astype(int)
national_df = national_df.dropna(subset=['vsby', 'tmpf', 'dwpf'])

national_df['t_td_spread'] = national_df['tmpf'] - national_df['dwpf']
national_df['wind_speed_mph'] = national_df['sknt'] * 1.15078

# Verify DC stations are not present
dc_in_training = national_df[national_df['station'].isin(DC_STATIONS)]
assert len(dc_in_training) == 0, f"DC stations found in training data! {dc_in_training['station'].unique()}"
print("DC holdout verified: no DC metro stations in training set.")

national_df.to_parquet(f'{BASE_DIR}/data/raw/asos/national_train_2018_2023.parquet')
print(f"Final training set: {len(national_df):,} rows")
print(f"Fog events: {national_df['is_fog'].sum():,} ({national_df['is_fog'].mean()*100:.2f}%)")

In [ ]:
# Step 4: Pull DC metro test data (Jan 2024 – present)

test_frames = []
for station in DC_STATIONS:
    print(f"Pulling {station}...")
    df = fetch_iem_asos(station, 2024, 2026)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)
test_df['valid'] = pd.to_datetime(test_df['valid'], utc=True)
test_df['vsby_km'] = test_df['vsby'] * 1.60934
test_df['is_fog'] = (test_df['vsby_km'] < 1.0).astype(int)
test_df = test_df.dropna(subset=['vsby', 'tmpf', 'dwpf'])
test_df['t_td_spread'] = test_df['tmpf'] - test_df['dwpf']
test_df['wind_speed_mph'] = test_df['sknt'] * 1.15078

test_df.to_parquet(f'{BASE_DIR}/data/raw/asos/dc_metro_test_2024_2026.parquet')
print(f"Test set: {len(test_df):,} rows, {test_df['is_fog'].sum()} fog events")

In [ ]:
# Step 5: Verify outputs
train_check = pd.read_parquet(f'{BASE_DIR}/data/raw/asos/national_train_2018_2023.parquet')
test_check  = pd.read_parquet(f'{BASE_DIR}/data/raw/asos/dc_metro_test_2024_2026.parquet')

train_check['valid'] = pd.to_datetime(train_check['valid'], utc=True)
test_check['valid']  = pd.to_datetime(test_check['valid'],  utc=True)

print(f"Training: {len(train_check):,} rows, date range {train_check['valid'].min().date()} – {train_check['valid'].max().date()}")
print(f"Test:     {len(test_check):,} rows,  date range {test_check['valid'].min().date()} – {test_check['valid'].max().date()}")

# Rule 3: temporal integrity
assert train_check['valid'].max() < test_check['valid'].min(), 'TEMPORAL LEAKAGE!'
print(f"\n✅ Temporal integrity OK")
print(f"DC stations NOT in training: {set(DC_STATIONS) - set(train_check['station'])}")
print(f"DC stations in test: {sorted(test_check['station'].unique())}")